# Multiview-Invariance Dataset — Usage Examples

This notebook shows two common workflows:

1. **Train / test split** — how to divide the dataset at the scene level so no related viewpoints leak across splits.
2. **Benchmarking loop** — how to shuffle and iterate the full dataset for evaluation.

Before running this notebook, make sure you have built the index:
```bash
python build_dataset_index.py
```

In [ ]:
from dataset import MultiviewDataset

## 1. Load the dataset

In [ ]:
ds = MultiviewDataset("dataset")
print(ds)

In [ ]:
# Quick peek at one group and its viewpoints
group, examples = next(ds.iter_groups())

print("Group:", group["group_id"])
print("  scene_id :", group["scene_id"])
print("  pair_id  :", group["pair_id"])
print("  object A :", group["object_A"])
print("  object B :", group["object_B"])
print("  viewpoints:", group["n_viewpoints"])
print()
for ex in examples:
    print("  example:", ex["example_id"])
    print("    image_path       :", ex["image_path"])
    print("    spatial_relations:", ex["spatial_relations"])
    print("    yaw_to_arrow     :", ex["yaw_to_arrow"])

---
## 2. Train / test split

The split unit is the **scene**.  
Every object pair and every viewpoint from a given scene lands entirely in either train or test — never both.

In [ ]:
train_ds, test_ds = ds.split_by_scene(train_frac=0.8, seed=42)

print("Train:", train_ds)
print("Test: ", test_ds)

In [ ]:
# Verify there is no scene overlap between the two splits
train_scenes = {s["scene_id"] for s in train_ds.scenes}
test_scenes  = {s["scene_id"] for s in test_ds.scenes}

overlap = train_scenes & test_scenes
print("Scenes in overlap (should be empty):", overlap)

In [ ]:
# Each split is a full MultiviewDataset — all the same methods work on it
print("First 3 train scenes:")
for s in train_ds.scenes[:3]:
    print(" ", s["scene_id"], "-", s["n_groups"], "groups")

print()
print("First 3 test scenes:")
for s in test_ds.scenes[:3]:
    print(" ", s["scene_id"], "-", s["n_groups"], "groups")

---
## 3. Benchmarking loop

For benchmarking you typically want to:
- shuffle so the model doesn't see scenes in a fixed order (some scenes may be easier)
- keep all viewpoints of each object pair together (they are the unit of comparison)

`shuffle_groups` does exactly this: it randomises the order of groups while keeping each group's viewpoints adjacent.

In [ ]:
# Shuffle the test set before benchmarking
test_ds.shuffle_groups(seed=0)

print("First 5 example IDs after shuffle:")
for ex in list(test_ds.iter_examples())[:5]:
    print(" ", ex["example_id"])

In [ ]:
# --- Benchmarking loop skeleton ---
#
# Replace `my_model_predict` with your actual VLM call.

def my_model_predict(image_path: str, question: str) -> dict:
    """Stub: replace with a real VLM call. Should return predicted relation flags."""
    # Return a dummy prediction for illustration
    return {
        "A_left_of_B":    False,
        "A_right_of_B":   True,
        "A_in_front_of_B": False,
        "A_behind_B":     True,
        "A_above_B":      False,
        "A_below_B":      True,
    }


results = []  # collect per-example results

for group, examples in test_ds.iter_groups():
    obj_a = group["object_A"]["label"]
    obj_b = group["object_B"]["label"]
    question = f"Is the {obj_a} to the left or right of the {obj_b}?"

    for ex in examples:
        prediction = my_model_predict(ex["image_path"], question)
        ground_truth = ex["spatial_relations"]

        # Check left/right accuracy as an example metric
        correct = (
            prediction["A_left_of_B"]  == ground_truth["A_left_of_B"] and
            prediction["A_right_of_B"] == ground_truth["A_right_of_B"]
        )

        results.append({
            "example_id": ex["example_id"],
            "group_id":   ex["group_id"],
            "correct":    correct,
        })

# Overall accuracy
accuracy = sum(r["correct"] for r in results) / len(results)
print(f"Left/right accuracy on test set: {accuracy:.1%}  ({len(results)} examples)")

In [ ]:
# Per-group consistency: what fraction of groups did the model get right
# on ALL viewpoints (the harder, invariance-focused metric)
from collections import defaultdict

group_results = defaultdict(list)
for r in results:
    group_results[r["group_id"]].append(r["correct"])

fully_correct_groups = sum(
    1 for scores in group_results.values() if all(scores)
)
total_groups = len(group_results)

group_consistency = fully_correct_groups / total_groups
print(f"Group consistency (all views correct): {group_consistency:.1%}  ({fully_correct_groups}/{total_groups} groups)")

---
## 4. Load a single group by ID

Useful when you want to inspect a specific pair or debug a prediction.

In [ ]:
# Pick any group_id from the dataset
sample_group_id = ds.groups[0]["group_id"]
print("Looking up group:", sample_group_id)

examples = ds.get_group_examples(sample_group_id)
for ex in examples:
    print("  view", ex["viewpoint_index"], "|", ex["image_path"])

In [ ]:
# Visualise the images for that group (requires PIL)
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(examples), figsize=(6 * len(examples), 5))
if len(examples) == 1:
    axes = [axes]

for ax, ex in zip(axes, examples):
    img = Image.open(ex["image_path"])
    ax.imshow(img)
    rels = ex["spatial_relations"]
    true_rels = [k for k, v in rels.items() if v]
    ax.set_title(f"view {ex['viewpoint_index']}\n" + "\n".join(true_rels), fontsize=9)
    ax.axis("off")

fig.suptitle(sample_group_id, fontsize=11)
plt.tight_layout()
plt.show()